In [1]:
import os
import pandas as pd

# Directory containing the .flow files
directory = 'ISCX_TOR/scale_60/'

# List to store individual dataframes
dataframes = []

# Loop through each file in the directory
for filename in os.listdir(directory):
    if filename.endswith('.flow'):
        # Read the file into a dataframe
        df = pd.read_csv(os.path.join(directory, filename))  # Adjust the reading method if necessary
        
        # Add a label column with the filename (without extension) as the label value
        df['label'] = os.path.splitext(filename)[0]
        
        # Append the dataframe to the list
        dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=True)

# Display the combined dataframe
print(combined_df)


      TotBytes  SrcBytes  DstBytes  SrcGap  DstGap  sMeanPktSz   dMeanPktSz  \
0       171119     18974    152145     0.0     0.0  208.505493  1094.568359   
1      4189477    221491   3967986     0.0     0.0  129.602692  1252.916382   
2          102        42        60     NaN     NaN   42.000000    60.000000   
3      3780934    197773   3583161     0.0     0.0  132.644531  1263.901611   
4      4892922    255279   4637643     0.0     0.0  131.451599  1263.318726   
...        ...       ...       ...     ...     ...         ...          ...   
1880   1248372    702438    545934     0.0     0.0  502.459229   313.035553   
1881   1148164    648132    500032     0.0     0.0  492.876038   306.392151   
1882   1245908    678392    567516     0.0     0.0  498.817657   318.829224   
1883    842887    467812    375075     0.0     0.0  484.277435   310.749786   
1884    107585     58537     49048     0.0     0.0  513.482483   338.262054   

      sMaxPktSz  dMaxPktSz  sMinPktSz  ...         

In [2]:
import os
import yaml
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import re

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

# Function to preprocess each dataset
def preprocess_dataset(df):
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Handle missing values for non-numeric data
        if not non_numeric_cols.empty:
            imputer_non_numeric = SimpleImputer(strategy='most_frequent')
            df_non_numeric = pd.DataFrame(imputer_non_numeric.fit_transform(df[non_numeric_cols]), columns=non_numeric_cols)
            # Convert non-numeric features to one-hot encoding
            encoder = OneHotEncoder(drop='first')
            encoded = encoder.fit_transform(df_non_numeric)
            encoded_df = pd.DataFrame(encoded.toarray(), columns=encoder.get_feature_names_out(non_numeric_cols))
        else:
            encoded_df = pd.DataFrame()  # Empty DataFrame for encoded non-numeric data if no non-numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric, encoded_df], axis=1)
        
        features = [
            'State__FA', 'State__A', 'pRetran', 'Max,sMeanPktSz', 'SrcRetra', 'PCRatio', 'State_PA_', 'State_PA_A',
            'SrcWin,SrcLoss', 'DstRate', 'SrcLoad', 'TcpOpt_MwsS  T', 'Load', 'DstLoad', 'TcpRtt', 'Flgs_ e g      ',
            'State_A_PA', 'Flgs_ e d      ', 'Sum', 'AckDat', 'dTtl', 'State__PA', 'Min', 'pLoss', 'DstLoss', 'State_S_',
            'Cause_Status', 'State_FA_', 'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts', 'Flgs_ e s      ',
            'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'State_S_SA', 'DstPkts', 'Flgs_ e *      ', 'Mean',
            'SrcBytes', 'State__SA', 'TotBytes', 'Cause_Start', 'dMeanPktSz', 'State_A_A', 'DstRetra', 'SynAck'
        ]

        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        # Scale features
        scaler = MinMaxScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df_preprocessed), columns=df_preprocessed.columns)

        return df_scaled
    else:
        return pd.DataFrame()
    
# Step 1: Separate the label column
X = combined_df.drop(columns=['label'])
y = combined_df['label']

pre_df = preprocess_dataset(X)

pre_df['label'] = y

print(len(pre_df.columns))
pre_df['label']

44


0       audio
1       audio
2       audio
3       audio
4       audio
        ...  
1880     voip
1881     voip
1882     voip
1883     voip
1884     voip
Name: label, Length: 1885, dtype: object

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report
import numpy as np

# Assuming `df` is your DataFrame and 'label' is your target column

# Step 1: Separate features (X) and target (y)
X = pre_df.drop(columns=['label'])  # Drop the label column to get the features
y = pre_df['label']  # Target variable

# Step 2: Initialize the Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Step 3: Perform cross-validation
cv_results = cross_validate(clf, X, y, cv=5, scoring=['accuracy', 'f1_macro', 'recall_macro'], return_train_score=False)

# Print cross-validation results
print(f"Cross-Validation Accuracy: {np.mean(cv_results['test_accuracy']):.4f} ± {np.std(cv_results['test_accuracy']):.4f}")
print(f"Cross-Validation F1 Score: {np.mean(cv_results['test_f1_macro']):.4f} ± {np.std(cv_results['test_f1_macro']):.4f}")
print(f"Cross-Validation Recall: {np.mean(cv_results['test_recall_macro']):.4f} ± {np.std(cv_results['test_recall_macro']):.4f}")

# Optionally, fit the model on the entire dataset for feature importance
clf.fit(X, y)

# Step 4: Feature Importances
importances = clf.feature_importances_
indices = importances.argsort()[::-1]  # Sort in descending order

# Step 5: Print top 10 feature importances
print("\nTop 10 Feature Importances:")
for i in range(min(10, len(indices))):  # Ensure there are at least 10 features
    print(f"Feature {X.columns[indices[i]]}: {importances[indices[i]]:.4f}")

# If you still want to compute confusion matrix and classification report, you need to split data again and train/test model
# For demonstration purposes, let's use the initial train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Confusion Matrix and Classification Report
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Cross-Validation Accuracy: 0.8398 ± 0.0376
Cross-Validation F1 Score: 0.7621 ± 0.0442
Cross-Validation Recall: 0.7637 ± 0.0358

Top 10 Feature Importances:
Feature IdleTime: 0.1958
Feature SrcLoad: 0.0630
Feature dMeanPktSz: 0.0531
Feature SrcBytes: 0.0526
Feature Load: 0.0468
Feature Rate: 0.0380
Feature Min: 0.0374
Feature DstRate: 0.0374
Feature TotBytes: 0.0364
Feature SrcRate: 0.0363

Confusion Matrix:
[[ 44   5   0   0   0   0   0   0]
 [  9  84   8   0   0   0   0   0]
 [  3  22  10   0   0   0   0   0]
 [  0   2   0  47   0   0   0   1]
 [  1   0   0   0  11   0   0   0]
 [  0   0   0   0   0  59   0   0]
 [  0   0   0   0   0   0 126   8]
 [  0   0   0   0   0   0   3 123]]

Classification Report:
              precision    recall  f1-score   support

       audio       0.77      0.90      0.83        49
    browsing       0.74      0.83      0.79       101
        chat       0.56      0.29      0.38        35
         ftp       1.00      0.94      0.97        50
        mail 

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Read the DataFrame (Replace this with your DataFrame reading code)
df = pre_df

# List of features to compute incremental statistics
feature_list = ['SynAck', 'TcpRtt', 'AckDat']  # Replace with your actual features

import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

def analyze_features(df):
    """
    Analyzes features of the DataFrame based on statistical properties and diversity.
    
    Parameters:
    - df: pandas DataFrame containing the features to analyze.
    
    Returns:
    - stats_df_sorted: pandas DataFrame containing the statistical metrics for each feature, sorted by variance.
    """
    # Separate features (assuming the last column is the target variable)
    X = df.drop(columns=[df.columns[-1]])  # Drop the last column assuming it's the target
    
    # Initialize a DataFrame to hold statistical metrics
    stats_df = pd.DataFrame(index=X.columns)
    
    # Calculate statistical metrics
    stats_df['Mean'] = X.mean()
    stats_df['Median'] = X.median()
    stats_df['StdDev'] = X.std()
    stats_df['Variance'] = X.var()
    stats_df['Range'] = X.max() - X.min()
    stats_df['Skewness'] = X.apply(lambda x: skew(x.dropna()))
    stats_df['Kurtosis'] = X.apply(lambda x: kurtosis(x.dropna()))
    stats_df['Missing Values'] = X.isna().sum()
    
    # Sort by variance (or any other metric you prefer)
    stats_df_sorted = stats_df.sort_values(by='Variance', ascending=False)
    
    # Print the sorted statistics
    print("Feature Statistical Analysis and Diversity:")
    print(stats_df_sorted)
    
    return stats_df_sorted

# Example usage:
# df = pd.read_csv('your_data.csv')  # Load your DataFrame
# stats_df_sorted = analyze_features(df)


# Compute incremental statistics
def compute_incremental_stats(df, features):
    incremental_stats = pd.DataFrame(index=df.index)
    
    for feature in features:
        incremental_stats[f'{feature}_mean'] = df[feature].expanding().mean()
        incremental_stats[f'{feature}_median'] = df[feature].expanding().median()
        incremental_stats[f'{feature}_std'] = df[feature].expanding().std()
        incremental_stats[f'{feature}_max'] = df[feature].expanding().max()
        incremental_stats[f'{feature}_min'] = df[feature].expanding().min()
    
    return incremental_stats

# Compute incremental statistics for the features
incremental_stats = compute_incremental_stats(df, feature_list)

# Fill NaN values in incremental statistics with 0
incremental_stats.fillna(0, inplace=True)

# Add incremental statistics to the original DataFrame
df_with_stats = pd.concat([df, incremental_stats], axis=1)

# Standardize the incremental features before clustering
incremental_feature_cols = [col for col in df_with_stats.columns if col.endswith(('mean', 'median', 'std', 'max', 'min'))]

X = df_with_stats[incremental_feature_cols]

analyze_features(X)

# Fill NaN values in the features (if any) with 0
X.fillna(0, inplace=True)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Perform MiniBatch KMeans clustering
n_clusters = 2
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to the DataFrame
df_with_stats['cluster'] = clusters

# Remove incremental features
df_final = df_with_stats.drop(columns=incremental_feature_cols)

# Initialize dictionary to store classifiers and results
classifiers = {}
results = {}

# Train and evaluate Random Forest Classifier for each cluster with cross-validation
for i in range(n_clusters):
    # Select data for the current cluster
    cluster_data = df_final[df_final['cluster'] == i]
    
    if len(cluster_data) == 0:
        print(f"Cluster {i} has no data. Skipping.")
        continue
    
    # Separate features and target
    X_cluster = cluster_data.drop(columns=['label'])
    y_cluster = cluster_data['label']
    
    # Check if there is enough data to perform cross-validation
    if len(X_cluster) < 5:
        print(f"Not enough data for cross-validation in cluster {i}.")
        continue
    
    # Initialize Random Forest Classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Perform cross-validation
    cv_results = cross_validate(clf, X_cluster, y_cluster, cv=5, scoring=['accuracy', 'f1_macro', 'recall_macro'], return_train_score=False)
    
    # Fit the model on the entire cluster data for confusion matrix
    clf.fit(X_cluster, y_cluster)
    
    # Predict on the same data to compute confusion matrix
    y_pred = clf.predict(X_cluster)
    
    # Compute confusion matrix
    cm = confusion_matrix(y_cluster, y_pred)
    
    # Store the classifier and results
    classifiers[i] = clf
    results[i] = {
        'cv_accuracy': np.mean(cv_results['test_accuracy']),
        'cv_f1_score': np.mean(cv_results['test_f1_macro']),
        'cv_recall': np.mean(cv_results['test_recall_macro']),
        'confusion_matrix': cm,
        'classification_report': classification_report(y_cluster, y_pred)
    }

# Print the results for each cluster
for i, result in results.items():
    print(f"\nCluster {i} - Random Forest Classifier Performance:")
    print(f"Cross-Validation Accuracy: {result['cv_accuracy']:.4f}")
    print(f"Cross-Validation F1 Score: {result['cv_f1_score']:.4f}")
    print(f"Cross-Validation Recall: {result['cv_recall']:.4f}")
    print("\nConfusion Matrix:")
    print(result['confusion_matrix'])
    print("\nClassification Report:")
    print(result['classification_report'])


Feature Statistical Analysis and Diversity:
                   Mean    Median    StdDev  Variance     Range  Skewness  \
AckDat_max     0.702931  0.551402  0.268084  0.071869  1.000000 -0.430225   
TcpRtt_max     0.943345  1.000000  0.211779  0.044850  1.000000 -4.210533   
SynAck_max     0.943059  1.000000  0.211739  0.044834  1.000000 -4.208916   
TcpRtt_std     0.189938  0.197780  0.050561  0.002556  0.275775 -2.354844   
SynAck_std     0.189882  0.197719  0.050548  0.002555  0.275716 -2.354659   
TcpRtt_mean    0.048531  0.048404  0.018698  0.000350  0.102736 -0.277447   
SynAck_mean    0.048516  0.048389  0.018693  0.000349  0.102715 -0.277234   
AckDat_std     0.059181  0.060775  0.014809  0.000219  0.081133 -2.888396   
AckDat_mean    0.013204  0.013014  0.004643  0.000022  0.024388 -0.461186   
SynAck_median  0.000000  0.000000  0.000000  0.000000  0.000000       NaN   
SynAck_min     0.000000  0.000000  0.000000  0.000000  0.000000       NaN   
TcpRtt_median  0.000000  0.00000

C:\Users\mjf\AppData\Local\Temp\ipykernel_19084\3948902731.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(0, inplace=True)



Cluster 0 - Random Forest Classifier Performance:
Cross-Validation Accuracy: 0.9330
Cross-Validation F1 Score: 0.9264
Cross-Validation Recall: 0.9330

Confusion Matrix:
[[147   0   0]
 [  0 337   0]
 [  0   0 413]]

Classification Report:
              precision    recall  f1-score   support

       audio       1.00      1.00      1.00       147
       video       1.00      1.00      1.00       337
        voip       1.00      1.00      1.00       413

    accuracy                           1.00       897
   macro avg       1.00      1.00      1.00       897
weighted avg       1.00      1.00      1.00       897


Cluster 1 - Random Forest Classifier Performance:
Cross-Validation Accuracy: 0.8026
Cross-Validation F1 Score: 0.7680
Cross-Validation Recall: 0.7711

Confusion Matrix:
[[ 51   0   0   0   0   0   0]
 [  0 308   1   0   0   0   0]
 [  0   0 101   0   0   0   0]
 [  0   0   0 167   0   0   0]
 [  0   0   0   0  55   0   0]
 [  0   0   0   0   0 197   0]
 [  0   0   0   0   0  